In [62]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

In [63]:
import os
from getpass import getpass

api_key = getpass("Cole sua OPENAI_API_KEY e pressione Enter: ").strip()
if not api_key:
    raise ValueError("Adicione sua OPENAI_API_KEY para continuar.")

os.environ["OPENAI_API_KEY"] = api_key

In [64]:
#load dos modelos
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    chunk_size=1000,
    timeout=60,
    show_progress_bar=True,
)
llm = ChatOpenAI(
    model="gpt-6-luna",
    max_completion_tokens=2000,
    reasoning_effort="low",
)

In [65]:
#load do arquivo
pdf_link = "os-sertoes.pdf"

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

In [66]:
len(pages)

658

In [67]:
#split dos documentos
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000, 
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

In [68]:
#Storage
import os
import chromadb
from chromadb.api.shared_system_client import SharedSystemClient

persist_directory = os.path.abspath("parent_rag_db")
os.makedirs(persist_directory, exist_ok=True)

# Chroma reuses one client per path in this process. After the folder is
# removed, that cached client can no longer open the sqlite file.
for system in list(SharedSystemClient._identifier_to_system.values()):
    try:
        system.stop()
    except Exception:
        pass
SharedSystemClient.clear_system_cache()

store = InMemoryStore()
vector_store = Chroma(
    client=chromadb.PersistentClient(path=persist_directory),
    embedding_function=embeddings_model,
    persist_directory=persist_directory,
)


In [69]:
parent_retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    docstore=store,
)

# Chroma rejects one upsert larger than its max batch size.
max_batch = vector_store._client.get_max_batch_size()
parents = parent_splitter.split_documents(pages)
batch = []
pending_children = 0
for doc in parents:
    n_children = max(1, len(child_splitter.split_documents([doc])))
    if batch and pending_children + n_children > max_batch:
        parent_retriever.add_documents(batch, ids=None)
        batch = []
        pending_children = 0
    batch.append(doc)
    pending_children += n_children
if batch:
    parent_retriever.add_documents(batch, ids=None)


  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

In [70]:
TEMPLATE = """
Você é um especialista em literatura brasileira. Responda a pergunta abaixo utilizando o contexto informado.
Query: {question}

Contexto:
{context}

Resposta:
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [71]:
setup_retrival = RunnableParallel({
    "question": RunnablePassthrough(),
    "context": parent_retriever
})

output_parser = StrOutputParser()

In [72]:
parent_chain_retrival = setup_retrival | rag_prompt | llm | output_parser

In [77]:
user_question = input("Digite sua pergunta: ")
parent_chain_retrival.invoke(user_question)

  0%|          | 0/1 [00:00<?, ?it/s]

'Em **_Os Sertões_**, a crítica social e política se concentra no contraste entre o Brasil litorâneo, ligado ao poder e à vida urbana, e o sertão, que permanece pobre, isolado e esquecido pelos governantes. Ao retratar a população de Canudos e as condições duras da região, Euclides da Cunha denuncia uma realidade marcada pela injustiça social e pela distância entre o país oficial e o Brasil profundo.\n\nA crítica política aparece sobretudo na **campanha de Canudos**: a repressão militar contra os sertanejos revela a violência do Estado e a incompreensão das elites diante de uma comunidade que não conheciam. O livro também questiona a ideia de que a sociedade urbana representava, por si só, a civilização, enquanto o sertão seria apenas barbárie.\n\nEssa posição torna a obra uma espécie de **mea culpa** da geração letrada à qual Euclides pertencia: o autor reconhece que ela não compreendia o interior do país. Isso não significa, porém, uma defesa da rejeição ao progresso — Euclides afirm